# Modelling — Heineken Demand Forecasting

Goal: predict weekly demand 8 weeks out, for each SKU × supermarket combination.

Quick roadmap:
1. Load and prep data
2. Build features (all lagged ≥ 8 weeks to avoid leakage)
3. Try a few models, compare on a time-based holdout
4. Look at what features are actually driving the predictions

## 1. Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

from utils_updated import (
    read_demand, read_promotions, extend_promotions_days,
    merge, clean_demand_per_group, aggregate_to_weekly,
    PROMOTION_DURATION_DAYS
)

np.random.seed(42)

## 2. Load & prep

In [ ]:
demand = read_demand("./demand.csv")
promotions = read_promotions("./promotions.csv")

def run_pipeline(demand, promotions):
    cleaned  = clean_demand_per_group(demand.copy())
    extended = extend_promotions_days(promotions, PROMOTION_DURATION_DAYS)
    daily    = merge(cleaned, extended.drop('promotion_id', axis=1))
    return aggregate_to_weekly(daily)

weekly = run_pipeline(demand, promotions)
print(f"Weekly data: {weekly.shape[0]} rows, {weekly.shape[1]} columns")
weekly.head(10)

## 3. Feature engineering

The hard constraint here is the 8-week forecast horizon — any feature using demand data must use values from at least t-8, otherwise we'd be peeking at future data during training and the model would look good on paper but fail in production.

Features:
- Lag 8, 9, 10, 12, 13 weeks — most recent values we can actually observe
- Lag 52 weeks — same week last year, picks up any annual pattern
- Rolling mean and std over 4, 8, 13 weeks — trend and volatility context
- Promotion flag — known ahead of time from the planning calendar
- Month, week of year, quarter — seasonality (mild here but worth keeping)
- SKU and store encoded — lets one model serve all 9 series

In [ ]:
def make_features(df):
    df = df.copy()

    df['year']        = df.index.year
    df['month']       = df.index.month
    df['week_of_year'] = df.index.isocalendar().week.astype(int)
    df['quarter']     = df.index.quarter
    df['promotion']   = df['promotion'].astype(int)

    chunks = []
    for (sku, sm), grp in df.groupby(['sku', 'supermarket']):
        grp = grp.sort_index()

        for lag in [8, 9, 10, 12, 13, 52]:
            grp[f'lag_{lag}w'] = grp['demand'].shift(lag)

        for w in [4, 8, 13]:
            shifted = grp['demand'].shift(8)
            grp[f'rolling_mean_{w}w'] = shifted.rolling(w).mean()
            grp[f'rolling_std_{w}w']  = shifted.rolling(w).std()

        chunks.append(grp)

    df = pd.concat(chunks)

    le_sku = LabelEncoder()
    le_sm  = LabelEncoder()
    df['sku_enc'] = le_sku.fit_transform(df['sku'])
    df['sm_enc']  = le_sm.fit_transform(df['supermarket'])

    return df


df_feat = make_features(weekly).dropna()
print(f"After feature engineering: {df_feat.shape}")
print(f"Remaining NaNs: {df_feat.isnull().sum().sum()}")

feature_cols = [
    'sku_enc', 'sm_enc', 'year', 'month', 'week_of_year', 'quarter', 'promotion',
    'lag_8w', 'lag_9w', 'lag_10w', 'lag_12w', 'lag_13w', 'lag_52w',
    'rolling_mean_4w', 'rolling_mean_8w', 'rolling_mean_13w',
    'rolling_std_4w', 'rolling_std_8w', 'rolling_std_13w'
]
print(f"\nFeatures: {len(feature_cols)}")

## 4. Train / test split

Can't use random splitting on time series — if test samples come from 2020 and training includes 2021, the model has implicitly seen the future. The split has to be purely chronological.

Holding out the last 8 weeks as the test set, which mirrors the actual deployment scenario (forecast the next 8 weeks).

In [ ]:
split_date = df_feat.index.max() - pd.Timedelta(weeks=8)

train = df_feat[df_feat.index <= split_date]
test  = df_feat[df_feat.index >  split_date]

X_train, y_train = train[feature_cols], train['demand']
X_test,  y_test  = test[feature_cols],  test['demand']

print(f"Train: {len(train):,} rows  ({train.index.min().date()} → {train.index.max().date()})")
print(f"Test:  {len(test):,} rows   ({test.index.min().date()} → {test.index.max().date()})")

## 5. Models

Before jumping into the ML models, I want a naive baseline to benchmark against. A model that can't beat "just take the recent average" isn't useful.

Then I'll try:
- **Ridge** — linear, fast, works well when the signal is mostly in the lag structure
- **Random Forest** — captures non-linear interactions, gives feature importance for free
- **Gradient Boosting / LightGBM** — typically strong on tabular data

One model across all 9 SKU×store series rather than 9 separate models. With only ~150 weeks per series, per-series models don't have much to learn from. A global model shares patterns (e.g. promotion effects) across all series.

In [ ]:
# naive baseline: predict the mean of the last 8 weeks for each series
naive_preds = []
for (sku, sm), grp in test.groupby(['sku', 'supermarket']):
    recent_mean = train[(train.sku == sku) & (train.supermarket == sm)]['demand'].tail(8).mean()
    naive_preds.extend([recent_mean] * len(grp))

naive_mape = mean_absolute_percentage_error(y_test, naive_preds) * 100
naive_mae  = mean_absolute_error(y_test, naive_preds)
print(f"Naive baseline — MAE: {naive_mae:.1f}, MAPE: {naive_mape:.1f}%")

In [ ]:
models = {
    'Ridge':             Ridge(alpha=10),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=8,
                                               min_samples_leaf=5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                   learning_rate=0.05, random_state=42),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                           num_leaves=31, random_state=42, verbose=-1),
}

results = {}
print(f"{'Model':<22} | {'MAE':>8} | {'MAPE':>8} | {'vs Naive':>10}")
print("-" * 57)

for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    mape  = mean_absolute_percentage_error(y_test, preds) * 100
    delta = (naive_mape - mape) / naive_mape * 100
    results[name] = {'MAE': mae, 'MAPE': mape, 'preds': preds, 'model': m}
    print(f"{name:<22} | {mae:>8.1f} | {mape:>7.1f}% | {delta:>+9.1f}%")

## 6. Evaluation

Using MAPE as the main metric — it's scale-free, so a 13% error means the same thing whether the series averages 100 units or 1000. MAE is in raw units which is useful for the business side but can't be compared across SKUs directly.

RMSE would penalise big misses more heavily, but for demand planning purposes a consistent 10% error is generally more useful to communicate than a metric that blows up on outlier weeks.

In [ ]:
best_name  = min(results, key=lambda k: results[k]['MAPE'])
best_preds = results[best_name]['preds']
best_model = results[best_name]['model']

print(f"Best model: {best_name} (MAPE = {results[best_name]['MAPE']:.1f}%)")

# per-series breakdown
test_copy = test.copy()
test_copy['pred'] = best_preds

rows = []
for (sku, sm), grp in test_copy.groupby(['sku', 'supermarket']):
    mape = mean_absolute_percentage_error(grp.demand, grp.pred) * 100
    mae  = mean_absolute_error(grp.demand, grp.pred)
    rows.append({'SKU': sku, 'Supermarket': sm, 'MAPE (%)': round(mape, 1), 'MAE': round(mae, 1)})

pd.DataFrame(rows).set_index(['SKU', 'Supermarket'])

In [ ]:
from IPython.display import Image
Image('model_fig1_comparison.png')

Ridge ends up winning, which is a bit surprising given that tree models usually do better on tabular data. I think it makes sense here though — once the lag features are constructed, the relationship between "what demand was 8 weeks ago" and "what demand is now" is mostly linear. The tree models overfit slightly on the small per-series samples.

Desperados × Jumbo is the hardest (highest MAPE) — consistent with the high CV we saw in the EDA.

## 7. Feature importance & promotion impact

In [ ]:
# use the Random Forest's built-in feature importances for this
# (Ridge coefficients aren't directly comparable across features with different scales)
rf = results['Random Forest']['model']
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Top 10 features:")
for feat, imp in fi.head(10).items():
    print(f"  {feat:<25}: {imp:.4f}")

promo_rank = list(fi.index).index('promotion') + 1
print(f"\n'promotion' ranks #{promo_rank} out of {len(feature_cols)}")

In [ ]:
# ablation: remove promotion and see how much MAPE goes up
feats_no_promo = [f for f in feature_cols if f != 'promotion']

rf_no_promo = RandomForestRegressor(n_estimators=200, max_depth=8,
                                     min_samples_leaf=5, random_state=42)
rf_no_promo.fit(X_train[feats_no_promo], y_train)
preds_no_promo = rf_no_promo.predict(X_test[feats_no_promo])

mape_with = results['Random Forest']['MAPE']
mape_without = mean_absolute_percentage_error(y_test, preds_no_promo) * 100

print(f"RF MAPE with promotion:    {mape_with:.2f}%")
print(f"RF MAPE without promotion: {mape_without:.2f}%")
print(f"Promotion adds:            {mape_without - mape_with:.2f} pp")

## 8. Assumptions and things I'd do differently with more time

**Assumptions:**
- Promotion calendar is known 8 weeks ahead — standard assumption in retail planning, but worth confirming
- Weekly granularity is sufficient — if daily forecasts are needed, the lag structure would change significantly
- No external data (weather, holidays, competitor activity)

**What I'd improve:**

| Thing | Impact | Effort |
|---|---|---|
| Walk-forward cross-validation | Better confidence on the MAPE estimate | Medium |
| Prediction intervals (quantile regression) | Lets planners set safety stock properly | Medium |
| National holiday calendar | Medium improvement | Low |
| Weather features (temp correlation with beer sales) | Probably high for Desperados | Medium |
| Proper hyperparameter tuning (Optuna) | Modest improvement | Medium |
| LightGBM with more tuning | Might match Ridge | Medium |

**Business framing:**

A 13% MAPE on Desperados (~587 units/week avg) means about ±76 units per week per store. That's the number a planner would use to set a safety stock buffer. Compared to the naive baseline at ~19%, this is a real improvement — both stock-outs and write-offs should come down.